# Gemma Psychological Distance Pipeline
**Thesis project – CMV Reddit annotations**

Uses `google/gemma-3-1b-it` as a zero-shot / few-shot classifier for:
- **Stage 1** – Salience detection
- **Stage 2** – Psychological distance annotation (4 dimensions)

> Run cells in order. Start with the **smoke test** to verify everything works, then run the **full evaluation**.

## 1. Install packages & login to HuggingFace

In [ ]:
!pip install -q transformers accelerate sentencepiece scikit-learn

from huggingface_hub import login
login(token='')  # Add your access token here

## 2. Verify GPU

Go to **Runtime → Change runtime type → T4 GPU** before running this cell.

In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU. Gemma will be very slow. Go to Runtime > Change runtime type.')

Device: cuda
GPU: Tesla T4
VRAM: 15.6 GB


## 3. Mount Google Drive & load data

Upload `train.json` and `dev.json` to your Google Drive and update the path below.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Update this path to match your Drive folder
DRIVE_BASE = '/content/drive/MyDrive'  # <-- change this
TRAIN_PATH = f'{DRIVE_BASE}/train.json'
DEV_PATH   = f'{DRIVE_BASE}/dev.json'
TEST_PATH  = f'{DRIVE_BASE}/test.json'
OUTPUT_DIR = f'{DRIVE_BASE}/gemma_results'

import os, json
os.makedirs(OUTPUT_DIR, exist_ok=True)

def load_data(path):
    with open(path) as f:
        return json.load(f)

train_data = load_data(TRAIN_PATH)
dev_data   = load_data(DEV_PATH)
test_data  = load_data(TEST_PATH)
print(f'Train: {len(train_data)} sentences')
print(f'Dev  : {len(dev_data)} sentences')
print(f'Test : {len(test_data)} sentences')

Mounted at /content/drive
Train: 723 sentences
Dev  : 176 sentences
Test : 155 sentences


## 4. Configuration

In [ ]:
MODEL_NAME      = 'google/gemma-3-1b-it'
DIMENSIONS      = ['temporal', 'spatial', 'social', 'hypothetical']
LABEL_MAP       = {-1: 0, 0: 1, 1: 2}
LABEL_INV       = {v: k for k, v in LABEL_MAP.items()}

print(f'Model : {MODEL_NAME}')
print(f'Device: {device}')

Model : google/gemma-3-1b-it
Device: cuda


## 5. Prompt templates

Prompts are based directly on the project annotation guidelines.
Gemma is used as a **few-shot classifier** — we give it the exact operational boundaries
and examples from the guidelines so it replicates the human annotation logic.

In [ ]:
SALIENCE_SYSTEM = """You are a research assistant specializing in Psycholinguistics and Construal Level Theory (CLT).
Your task is to analyze sentences from ChangeMyView (CMV) Reddit posts.

Classify each sentence as SALIENT or NOT_SALIENT.

SALIENT means the sentence contains argumentative content:
  - A Claim: the main point or opinion being argued
  - Evidence: facts, statistics, personal anecdotes, or events supporting a claim
  - Reasoning: the logical connection between a claim and evidence

NOT_SALIENT means the sentence is:
  - Meta-talk about the conversation itself (e.g. 'I read your post', 'Let me begin by saying')
  - A filler, greeting, or acknowledgement with no argumentative content
  - A pure transition with no claim (e.g. 'Therefore the logic follows that...')

IMPORTANT: Sentences that make factual claims, describe events, express opinions, or
present personal experiences as evidence ARE salient - even if they are short or informal.
Do not confuse informal language or negative phrasing with non-salience.

Respond with only one word: SALIENT or NOT_SALIENT"""

SALIENCE_FEW_SHOT = """Examples from the annotation guidelines:

Sentence: "My brother lost his job at the factory last Tuesday."
Answer: SALIENT

Sentence: "History shows that empires always eventually collapse."
Answer: SALIENT

Sentence: "If the government provided UBI, poverty would vanish."
Answer: SALIENT

Sentence: "I am not saying they don't have legitimate grievances."
Answer: SALIENT

Sentence: "Brown robbed a store and disobeyed a police officer."
Answer: SALIENT

Sentence: "I read your post earlier."
Answer: NOT_SALIENT

Sentence: "Before I begin, let me acknowledge your point."
Answer: NOT_SALIENT

Sentence: "But this post is about the Michael Brown case specifically."
Answer: NOT_SALIENT

Now classify the following sentence:
Sentence: "{sentence}"
Answer:"""


# ── Stage 2: Dimensions ───────────────────────────────────────────────────────
# Adapted directly from the project annotation prompt.
# The vector_explanation field forces chain-of-thought reasoning per dimension,
# which is critical for hypothetical where shortcutting causes most errors.

DIMENSION_SYSTEM = """You are a research assistant specializing in Psycholinguistics and Construal Level Theory (CLT).
Your task is to assign a psychological distance vector to a sentence from a CMV Reddit post.

For each dimension assign exactly one value: -1, 0, or 1.
Scoring:
  0 (Near): concrete, specific, low-level construal
  1 (Far): abstract, categorical, high-level construal
 -1 (N/A): no cue present for this dimension

## Operational boundaries (apply strictly)

TEMPORAL:
  -1 = No time reference; laws, norms, or logic with no stated timeframe. DEFAULT if no explicit time word.
   0 = Explicit time anchor within 1 month ('today', 'last week', 'right now', 'currently happening')
   1 = Explicit time anchor over 1 month or open-ended ('in the 1990s', 'for centuries', 'always', 'over the next decade')
  CRITICAL: Laws, policies, institutions, and general norms have NO inherent temporal distance.
  Ask yourself: does this sentence contain a time word or phrase? If not -> -1.
  Examples: 'The law requires X' -> -1 | 'The law passed last month' -> 0 | 'This has been the law for decades' -> 1

HYPOTHETICAL:
  -1 = Direct personal experience or first-hand observation ('I did X', 'I saw Y', 'I personally witnessed')
   0 = Asserted facts, historical events, established findings stated as true ('studies show', 'X happened in 1990', 'data shows')
   1 = Possibilities, conditionals, what-ifs, ideals, speculation ('if X then Y', 'suppose', 'ideally', 'could', 'imagine if', 'would')
  CRITICAL: This captures whether content is presented as real vs. possible/imagined.
  It is NOT about whether the sentence sounds abstract or philosophical.
  Ask yourself: is the author asserting this as true (-> 0), imagining a scenario (-> 1), or reporting personal experience (-> -1)?
  Examples: 'Corporations lobby for profit' -> 0 | 'If corporations were regulated they would lobby less' -> 1 | 'I personally witnessed this' -> -1

SPATIAL:
  -1 = No location reference
   0 = Local/travelable ('my house', 'this city', 'our neighborhood', 'the shop')
   1 = National/global/universal ('the EU', 'overseas', 'across the country', 'globally')

SOCIAL:
  -1 = No social reference
   0 = 'I', specific named individuals ('I', 'Elon Musk', 'my brother')
   1 = Groups, demographics, institutions ('the government', 'society', 'the police', 'women', 'immigrants')

## Output format
Return ONLY a valid JSON object. No prose, no markdown fences.
You MUST include a vector_explanation for each dimension — this forces you to reason before answering.

{
  \"vector\": {\"temporal\": <-1,0,1>, \"spatial\": <-1,0,1>, \"social\": <-1,0,1>, \"hypothetical\": <-1,0,1>},
  \"vector_explanation\": {
    \"temporal\": \"<your reasoning>\",
    \"spatial\": \"<your reasoning>\",
    \"social\": \"<your reasoning>\",
    \"hypothetical\": \"<your reasoning>\"
  }
}"""

DIMENSION_FEW_SHOT = """Examples:
Sentence: "Over the past couple of thousand years human beings as a species have done a pretty crappy job of living up to the 3 laws we expect robots to abide by: A robot may not injure a human being or, through inaction, allow a human being to come to harm."
Answer: {{"vector": {{"temporal": 1, "spatial": -1, "social": 1, "hypothetical": 0}}, "vector_explanation": {{"temporal": "Far (1) - over the past couple of thousand years spans millennia, well beyond 1 month.", "spatial": "N/A (-1) - no location cue.", "social": "Far (1) - human beings as a species is a categorical group.", "hypothetical": "Near (0) - stated as a historical observation, not a conditional."}}}}

Sentence: "Lightning will strike me if I stay out in the rain."
Answer: {{"vector": {{"temporal": -1, "spatial": 0, "social": -1, "hypothetical": 1}}, "vector_explanation": {{"temporal": "N/A (-1) - no explicit time word present.", "spatial": "Near (0) - out in the rain implies a local, immediately reachable setting.", "social": "N/A (-1) - me refers to the speaker as an individual but no social group is invoked.", "hypothetical": "Far (1) - if...will is a direct conditional/what-if scenario."}}}}

Sentence: "Yeah, for the night, everyone is drunk and having the time of their lives and the party is amazing but then the next morning comes around images of half naked fairies prancing around bombard the news."
Answer: {{"vector": {{"temporal": 0, "spatial": -1, "social": 0, "hypothetical": -1}}, "vector_explanation": {{"temporal": "Near (0) - for the night / the next morning are within a single day, under 1 month.", "spatial": "N/A (-1) - no location cue.", "social": "Near (0) - everyone is used colloquially about people at a specific event, not a broad social category.", "hypothetical": "N/A (-1) - described as a concrete, observed sequence of events."}}}}

Sentence: "IE The robots will treat all humans everywhere the same way we (as a species) routinely treated some humans in some places.* They won't want to co-exist but rather dominate."
Answer: {{"vector": {{"temporal": -1, "spatial": 1, "social": 1, "hypothetical": 0}}, "vector_explanation": {{"temporal": "N/A (-1) - no explicit time word present.", "spatial": "Far (1) - everywhere / some places implies a global or at minimum national scope.", "social": "Far (1) - all humans / we as a species is a categorical group.", "hypothetical": "Near (0) - framed as a predictive assertion based on historical pattern, not an explicit conditional."}}}}

Sentence: "This is the biggest snowfall in 15 years."
Answer: {{"vector": {{"temporal": 0, "spatial": 0, "social": -1, "hypothetical": 1}}, "vector_explanation": {{"temporal": "Near (0) - in 15 years provides a specific historical reference point within a bounded period.", "spatial": "Near (0) - implies a local or regional area the speaker is situated in.", "social": "N/A (-1) - no reference to any person or group.", "hypothetical": "Far (1) - the implied comparison (bigger than anything in 15 years) projects a counterfactual baseline."}}}}

Sentence: "Most European countries have had universal healthcare for a while now."
Answer: {{"vector": {{"temporal": 1, "spatial": 1, "social": 1, "hypothetical": -1}}, "vector_explanation": {{"temporal": "Far (1) - for a while now indicates an extended, open-ended duration beyond 1 month.", "spatial": "Far (1) - most European countries is national/continental in scope.", "social": "Far (1) - most European countries refers to institutions/governments as a categorical group.", "hypothetical": "N/A (-1) - stated as an established fact."}}}}

Sentence: "Let me compare it to the jury system in the US: normally a jury of the defendant's peers makes the decision on guilt or innocence."
Answer: {{"vector": {{"temporal": -1, "spatial": 1, "social": 0, "hypothetical": 0}}, "vector_explanation": {{"temporal": "N/A (-1) - no time word present.", "spatial": "Far (1) - in the US is national in scope.", "social": "Near (0) - a jury of the defendant's peers refers to a specific procedural body, not a broad social category.", "hypothetical": "Near (0) - described as a standard procedural fact, not a conditional."}}}}

Sentence: "Your first in line waiting at an intersection and as the light turns green you pull out then I run the light crossing your oncoming lane and taking the front left off your car, totalling it."
Answer: {{"vector": {{"temporal": 0, "spatial": -1, "social": 0, "hypothetical": 1}}, "vector_explanation": {{"temporal": "Near (0) - as the light turns green situates the event in an immediate, present-tense moment.", "spatial": "N/A (-1) - an intersection is implied local but no named location is given.", "social": "Near (0) - you and I refer to specific individuals.", "hypothetical": "Far (1) - the scenario is a hypothetical illustration constructed to make an argument."}}}}

Now annotate the following sentence:
Sentence: "{sentence}"
Answer:"""

print('Prompt templates defined (based on annotation prompt).')


Prompt templates defined (based on annotation prompt).


## 6. Load Gemma model

This downloads ~2.5GB — should take about 1-2 minutes on Colab.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

print(f'Loading {MODEL_NAME} ...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map='auto',
)
model.eval()
print(f'Model loaded on: {next(model.parameters()).device}')
print(f'VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB')

Loading google/gemma-3-1b-it ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

Model loaded on: cuda:0
VRAM used: 2.00 GB


## 7. Inference helpers

In [ ]:
import re
import numpy as np
from sklearn.metrics import classification_report, f1_score

def generate_response(prompt_system, prompt_user, max_new_tokens=60):
    messages = [{'role': 'user', 'content': f'{prompt_system}\n\n{prompt_user}'}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


def predict_salience(sentence):
    prompt = SALIENCE_FEW_SHOT.format(sentence=sentence)
    response = generate_response(SALIENCE_SYSTEM, prompt, max_new_tokens=15)
    response_upper = response.upper().strip()
    if 'NOT_SALIENT' in response_upper or 'NOT SALIENT' in response_upper:
        return False, response
    elif 'SALIENT' in response_upper:
        return True, response
    else:
        if any(w in response_upper for w in ['NO', 'FALSE', 'NON-SALIENT']):
            return False, response
        print(f'  WARNING: ambiguous salience response: "{response}" -> defaulting to SALIENT')
        return True, response


def parse_json_response(response):
    """
    Robustly extract the dimension vector from the model response.
    The new prompt returns a nested format:
      {"vector": {"temporal": ..., ...}, "vector_explanation": {...}}
    We extract the vector field. Falls back to flat {"temporal": ...} format.
    Also handles markdown fences and trailing reasoning text.
    """
    # Step 1: strip markdown fences
    cleaned = re.sub(r'```(?:json)?', '', response).strip()

    # Step 2: try to find and parse the outermost JSON object
    # Use a broader match to capture nested braces
    match = re.search(r'\{.*\}', cleaned, re.DOTALL)
    if match:
        try:
            parsed = json.loads(match.group())
            # New format: nested under 'vector' key
            if 'vector' in parsed:
                return parsed['vector']
            # Old flat format: {"temporal": ..., "spatial": ...}
            if 'temporal' in parsed:
                return parsed
        except (json.JSONDecodeError, ValueError):
            pass

    # Step 3: fall back to finding first flat {...} block
    match = re.search(r'\{[^{}]+\}', cleaned)
    if match:
        try:
            parsed = json.loads(match.group())
            if 'temporal' in parsed:
                return parsed
        except (json.JSONDecodeError, ValueError):
            pass

    return None


def predict_dimensions(sentence):
    prompt = DIMENSION_FEW_SHOT.format(sentence=sentence)
    # Increase max_new_tokens to accommodate reasoning in vector_explanation
    response = generate_response(DIMENSION_SYSTEM, prompt, max_new_tokens=300)
    parsed = parse_json_response(response)
    if parsed is not None:
        vector = {dim: int(parsed.get(dim, -1)) for dim in DIMENSIONS}
        vector = {dim: v if v in (-1, 0, 1) else -1 for dim, v in vector.items()}
        return vector, response
    print(f'  WARNING: could not parse dimension response: "{response[:100]}" -> defaulting to all -1')
    return {dim: -1 for dim in DIMENSIONS}, response


def filter_salient(data):
    return [d for d in data if d['salient'] and all(d.get(dim) is not None for dim in DIMENSIONS)]

print('Inference helpers defined.')

Inference helpers defined.


## 8. Smoke test

Runs on **5 sentences only** (4 salient + 1 non-salient) to verify the full pipeline works end-to-end.
Check that:
- The model produces a response for each sentence
- Stage 1 outputs SALIENT or NOT_SALIENT
- Stage 2 outputs a valid JSON vector
- No errors or crashes

If this looks good, proceed to the full evaluation in the next cells.

In [ ]:
# Select a small mix of salient and non-salient sentences
salient_sample     = [d for d in dev_data if d['salient']][:4]
non_salient_sample = [d for d in dev_data if not d['salient']][:1]
smoke_data         = salient_sample + non_salient_sample

print(f'Smoke test: {len(smoke_data)} sentences')
print('=' * 60)

for i, d in enumerate(smoke_data):
    sentence = d['sentence_text']
    true_sal = d['salient']

    # Stage 1
    pred_sal, raw_sal = predict_salience(sentence)

    # Stage 2 (only if salient)
    pred_dims, raw_dims = None, None
    if pred_sal and all(d.get(dim) is not None for dim in DIMENSIONS):
        pred_dims, raw_dims = predict_dimensions(sentence)

    print(f'\nSentence {i+1}: "{sentence[:80]}..."')
    print(f'  True salient : {true_sal}')
    print(f'  Pred salient : {pred_sal}  (raw: "{raw_sal}")')
    if pred_dims:
        true_dims = {dim: d.get(dim) for dim in DIMENSIONS}
        print(f'  True dims    : {true_dims}')
        print(f'  Pred dims    : {pred_dims}  (raw: "{raw_dims[:80]}")')

print('\n' + '=' * 60)
print('SMOKE TEST COMPLETE - if results look sensible, run the full evaluation below.')

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Smoke test: 5 sentences

Sentence 1: "Before I begin, let me acknowledge that there are very legitimate concerns and g..."
  True salient : True
  Pred salient : True  (raw: "SALIENT")
  True dims    : {'temporal': -1, 'spatial': -1, 'social': 1, 'hypothetical': 0}
  Pred dims    : {'temporal': -1, 'spatial': 1, 'social': 1, 'hypothetical': 0}  (raw: "{"vector": {"temporal": -1, "spatial": 1, "social": 1, "hypothetical": 0}, "vect")

Sentence 2: "I am not saying at all that they're aren't any legitimate grievances blacks hae;..."
  True salient : True
  Pred salient : True  (raw: "SALIENT")
  True dims    : {'temporal': -1, 'spatial': -1, 'social': -1, 'hypothetical': 0}
  Pred dims    : {'temporal': 0, 'spatial': 0, 'social': 0, 'hypothetical': 1}  (raw: "{"vector": {"temporal": 0, "spatial": 0, "social": 0, "hypothetical": 1}, "vecto")

Sentence 3: "As has been repeatedly established, Darren Wilson shot Brown in self-defense...."
  True salient : True
  Pred salient : True  (raw: "SA

## 9. Full evaluation – Stage 1 (Salience)

Runs on the full dev set (176 sentences). Expect ~10-20 minutes on a T4 GPU.

In [ ]:
print('=' * 60)
print('STAGE 1 - Salience  |  model: Gemma')
print('=' * 60)

all_preds_sal, all_labels_sal, sal_results = [], [], []

for i, d in enumerate(dev_data):
    pred, raw = predict_salience(d['sentence_text'])
    label     = int(d['salient'])
    all_preds_sal.append(int(pred))
    all_labels_sal.append(label)
    sal_results.append({'sentence_text': d['sentence_text'], 'true': bool(label), 'pred': pred, 'raw': raw})

    if (i + 1) % 20 == 0:
        f1_so_far = f1_score(all_labels_sal, all_preds_sal, average='macro')
        print(f'  {i+1}/{len(dev_data)} sentences processed  (running macro-F1: {f1_so_far:.4f})')

# Final evaluation
sal_f1 = f1_score(all_labels_sal, all_preds_sal, average='macro')
print(f'\nFinal dev macro-F1: {sal_f1:.4f}')
print(classification_report(all_labels_sal, all_preds_sal, target_names=['non-salient', 'salient'], digits=4))

# Save to Drive
sal_out = os.path.join(OUTPUT_DIR, 'salience_predictions.json')
with open(sal_out, 'w') as f:
    json.dump(sal_results, f, indent=2)
print(f'Predictions saved to {sal_out}')

STAGE 1 - Salience  |  model: Gemma
  20/176 sentences processed  (running macro-F1: 0.8039)
  40/176 sentences processed  (running macro-F1: 0.7403)
  60/176 sentences processed  (running macro-F1: 0.7115)
  80/176 sentences processed  (running macro-F1: 0.7949)
  100/176 sentences processed  (running macro-F1: 0.7629)
  120/176 sentences processed  (running macro-F1: 0.6773)
  140/176 sentences processed  (running macro-F1: 0.6788)
  160/176 sentences processed  (running macro-F1: 0.6691)

Final dev macro-F1: 0.6562
              precision    recall  f1-score   support

 non-salient     0.4000    0.4828    0.4375        29
     salient     0.8936    0.8571    0.8750       147

    accuracy                         0.7955       176
   macro avg     0.6468    0.6700    0.6562       176
weighted avg     0.8123    0.7955    0.8029       176

Predictions saved to /content/drive/MyDrive/gemma_results/salience_predictions.json


## 10. Full evaluation – Stage 2 (Dimensions)

Runs on salient sentences only (~147 sentences). Expect ~20-30 minutes on a T4 GPU.

In [ ]:
print('=' * 60)
print('STAGE 2 - Dimensions  |  model: Gemma')
print('=' * 60)

salient_dev = filter_salient(dev_data)
print(f'Salient sentences: {len(salient_dev)}')

all_preds_dim  = {dim: [] for dim in DIMENSIONS}
all_labels_dim = {dim: [] for dim in DIMENSIONS}
dim_results    = []

for i, d in enumerate(salient_dev):
    pred_vector, raw = predict_dimensions(d['sentence_text'])

    for dim in DIMENSIONS:
        all_preds_dim[dim].append(LABEL_MAP.get(pred_vector[dim], 0))
        all_labels_dim[dim].append(LABEL_MAP.get(d[dim], 0))

    dim_results.append({
        'sentence_text': d['sentence_text'],
        'true': {dim: d[dim] for dim in DIMENSIONS},
        'pred': pred_vector,
        'raw':  raw,
    })

    if (i + 1) % 20 == 0:
        print(f'  {i+1}/{len(salient_dev)} sentences processed...')

# Per-dimension evaluation
per_dim_f1 = []
for dim in DIMENSIONS:
    f1 = f1_score(all_labels_dim[dim], all_preds_dim[dim], average='macro', zero_division=0)
    per_dim_f1.append(f1)
    print(f'\n-- {dim.upper()} --')
    print(classification_report(
        all_labels_dim[dim], all_preds_dim[dim],
        target_names=['n/a (-1)', 'concrete (0)', 'abstract (1)'],
        digits=4, zero_division=0,
    ))

avg_f1 = float(np.mean(per_dim_f1))
print(f'\nAverage macro-F1 across dimensions: {avg_f1:.4f}')
print('Per dimension:', {dim: round(f1, 4) for dim, f1 in zip(DIMENSIONS, per_dim_f1)})

# Save to Drive
dim_out = os.path.join(OUTPUT_DIR, 'dimension_predictions.json')
with open(dim_out, 'w') as f:
    json.dump(dim_results, f, indent=2)
print(f'Predictions saved to {dim_out}')

STAGE 2 - Dimensions  |  model: Gemma
Salient sentences: 147
  20/147 sentences processed...
  40/147 sentences processed...
  60/147 sentences processed...
  80/147 sentences processed...
  100/147 sentences processed...
  120/147 sentences processed...
  140/147 sentences processed...

-- TEMPORAL --
              precision    recall  f1-score   support

    n/a (-1)     0.9310    0.9926    0.9609       136
concrete (0)     0.0000    0.0000    0.0000         5
abstract (1)     0.5000    0.1667    0.2500         6

    accuracy                         0.9252       147
   macro avg     0.4770    0.3864    0.4036       147
weighted avg     0.8818    0.9252    0.8992       147


-- SPATIAL --
              precision    recall  f1-score   support

    n/a (-1)     0.9310    0.9854    0.9574       137
concrete (0)     0.0000    0.0000    0.0000         1
abstract (1)     0.0000    0.0000    0.0000         9

    accuracy                         0.9184       147
   macro avg     0.3103    0

## 11. Results summary

In [ ]:
print('=' * 60)
print('GEMMA RESULTS SUMMARY')
print('=' * 60)
print(f'Stage 1 - Salience macro-F1 : {sal_f1:.4f}')
print(f'Stage 2 - Avg dimension F1  : {avg_f1:.4f}')
print()
print('Per dimension:')
for dim, f1 in zip(DIMENSIONS, per_dim_f1):
    print(f'  {dim:<14}: {f1:.4f}')
print()
print('Comparison with fine-tuned models (Stage 1):')
print('  DeBERTa : 0.8421')
print('  SBERT   : 0.9151')
print(f'  Gemma   : {sal_f1:.4f}')
print()
print('Comparison with fine-tuned models (Stage 2 avg):')
print('  DeBERTa : 0.3499')
print('  SBERT   : 0.2887')
print(f'  Gemma   : {avg_f1:.4f}')

GEMMA RESULTS SUMMARY
Stage 1 - Salience macro-F1 : 0.6562
Stage 2 - Avg dimension F1  : 0.3019

Per dimension:
  temporal      : 0.4036
  spatial       : 0.3191
  social        : 0.2779
  hypothetical  : 0.2068

Comparison with fine-tuned models (Stage 1):
  DeBERTa : 0.8421
  SBERT   : 0.9151
  Gemma   : 0.6562

Comparison with fine-tuned models (Stage 2 avg):
  DeBERTa : 0.3499
  SBERT   : 0.2887
  Gemma   : 0.3019


In [ ]:
# ── Final test set evaluation ──────────────────────────────
# Run ONCE after all dev evaluation is complete.
# Mirrors the structure of the BERT notebook Section 16.

print('=' * 60)
print('FINAL TEST SET EVALUATION  |  model: Gemma')
print('=' * 60)

# Load test data (add this line to Section 3 if not already there)
test_data = load_data(f'{DRIVE_BASE}/test.json')
print(f'Test: {len(test_data)} sentences')

# ── Stage 1: Salience ──
print('\n── Stage 1: Salience ──')

all_preds_sal, all_labels_sal, sal_results = [], [], []

for i, d in enumerate(test_data):
    pred, raw = predict_salience(d['sentence_text'])
    label     = int(d['salient'])
    all_preds_sal.append(int(pred))
    all_labels_sal.append(label)
    sal_results.append({'sentence_text': d['sentence_text'], 'true': bool(label), 'pred': pred, 'raw': raw})

    if (i + 1) % 20 == 0:
        f1_so_far = f1_score(all_labels_sal, all_preds_sal, average='macro')
        print(f'  {i+1}/{len(test_data)} sentences processed  (running macro-F1: {f1_so_far:.4f})')

sal_f1 = f1_score(all_labels_sal, all_preds_sal, average='macro')
print(f'\nTest macro-F1 (salience): {sal_f1:.4f}')
print(classification_report(all_labels_sal, all_preds_sal, target_names=['non-salient', 'salient'], digits=4))

# ── Stage 2: Dimensions ──
print('\n── Stage 2: CLT Dimensions ──')

salient_test   = filter_salient(test_data)
all_preds_dim  = {dim: [] for dim in DIMENSIONS}
all_labels_dim = {dim: [] for dim in DIMENSIONS}
dim_results    = []

for i, d in enumerate(salient_test):
    pred_vector, raw = predict_dimensions(d['sentence_text'])

    for dim in DIMENSIONS:
        all_preds_dim[dim].append(LABEL_MAP.get(pred_vector[dim], 0))
        all_labels_dim[dim].append(LABEL_MAP.get(d[dim], 0))

    dim_results.append({
        'sentence_text': d['sentence_text'],
        'true': {dim: d[dim] for dim in DIMENSIONS},
        'pred': pred_vector,
        'raw':  raw,
    })

    if (i + 1) % 20 == 0:
        print(f'  {i+1}/{len(salient_test)} sentences processed...')

per_dim_f1 = []
for dim in DIMENSIONS:
    f1 = f1_score(all_labels_dim[dim], all_preds_dim[dim], average='macro', zero_division=0)
    per_dim_f1.append(f1)
    print(f'\n-- {dim.upper()} --')
    print(classification_report(
        all_labels_dim[dim], all_preds_dim[dim],
        target_names=['n/a (-1)', 'concrete (0)', 'abstract (1)'],
        digits=4, zero_division=0,
    ))

avg_f1 = float(np.mean(per_dim_f1))
print(f'\nTest avg macro-F1 (dimensions): {avg_f1:.4f}')
for dim, f1 in zip(DIMENSIONS, per_dim_f1):
    print(f'  {dim:<14}: F1={f1:.4f}')

# Save predictions to Drive
sal_out = os.path.join(OUTPUT_DIR, 'salience_predictions_test.json')
dim_out = os.path.join(OUTPUT_DIR, 'dimension_predictions_test.json')
with open(sal_out, 'w') as f:
    json.dump(sal_results, f, indent=2)
with open(dim_out, 'w') as f:
    json.dump(dim_results, f, indent=2)
print(f'\nPredictions saved to {OUTPUT_DIR}')

FINAL TEST SET EVALUATION  |  model: Gemma
Test: 155 sentences

── Stage 1: Salience ──
  20/155 sentences processed  (running macro-F1: 0.6416)
  40/155 sentences processed  (running macro-F1: 0.6011)
  60/155 sentences processed  (running macro-F1: 0.5765)
  80/155 sentences processed  (running macro-F1: 0.5707)
  100/155 sentences processed  (running macro-F1: 0.5110)
  120/155 sentences processed  (running macro-F1: 0.5280)
  140/155 sentences processed  (running macro-F1: 0.5356)

Test macro-F1 (salience): 0.5333
              precision    recall  f1-score   support

 non-salient     0.2414    0.5000    0.3256        28
     salient     0.8557    0.6535    0.7411       127

    accuracy                         0.6258       155
   macro avg     0.5485    0.5768    0.5333       155
weighted avg     0.7447    0.6258    0.6660       155


── Stage 2: CLT Dimensions ──
  20/127 sentences processed...
  40/127 sentences processed...
  60/127 sentences processed...
  80/127 sentences pro